# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJztfYt220aS6K905JuITCjqYTuTaFazV5ZoWxNZ8oqys3tMXRyQbEqIQICDhxSOVv9+q6q7gX4BpOTszN49F3MmFoF+VlfXu6sfNiZpMouu+4vlxj7biOaLNCtYmo8S+WfGqz/LMppWP4poXn/Jb8oiiutvNxkPp1FyPUpmWTpnxXIBP5j8er4oojQJ4x47jfKix46jCfz3MFnK0ouwuImjsSr+EX6OklHygh3zgk8KNkjuoixN5jwp9tm7NL2OOTtK43DM7nJ2mk5CGMjJMDg6Pz18ww5gLn1ssc9/h97yzmhjG2ZcQOXRRpeFyZQlaeEWOtof4bPRxa6nfMaCyQ2f3AZRHkzT+yROwynPgmmUdSbQRjQNC46/9lleZF229Rc2TtN4f5QweEYb+L/PPItmEc/ZfVTcsOMsuuPs7afhgE1CaJpFyV0YYzsAG3Z/w4sbnrGQQZsw5zRbsiiHn1SG1SNg92l2my/CCe+LXkSPRbaUfeMTzWiOxkBZmhkTj3JnLl2tCXwyXpRZwt6Gcc7rLy/YmzIvxEzqwcYARlp/VcwcET6wAhmC44B94f04vedZp8tmMCyEBY4L23AHdVW3wn+f8AXgA/0DYFtrvPrId/vsCFeVuoUedHDPw+yWZ3ldHL8HCw2hfkujxBxcD5YaixHaaNBXb3FiatqIeSbwRQc21F3AVYNZG4Ky5StrAdrB5+vGaoGm9reSl7w/HYvpGTVgKKONOS9CfNv/LU8Tt5CnV7lwl1nJzY+TGFANtp2zCmKCCH5ZxFiBaqjVR2ek7mpUfdkL0rwo+hDXx+6qnyu3xTWWqKFTT2MEgnmIJbW1sOvSmiFVQeT3lWgYg7Fo+ibb0zdZXoTXQBe2Jzdlcpu7m4zGKAut2i/t+7BqpNtAx9yRvtRHijwLWAmb82kUsrwcz9IYCG7OOll434NGZlHCp+Y+D5Nl51YfNDZEL76MNqDaaANHJquONq7WHtkr38gWWfob8kMJRNYBes6B7dzm7qh4nyfTHDkP8LZ+tFgmY2R/FULKAa85IJeytiCqWVgw04znaXzHbVZKjBMYqME3xd/HyySc47RjYC7A4wuAruT9gpVqPLEi4j1YtwUKESh95Dfwx6Qs8p5osa9ebBVhds2LfGu83ALmSmMrcRI9Qrr8Jsz4lKnlv7+JoM/rMsxCkCI4tjyDNxKHoSg7G3weXLD7LCpAypCDgVEXqTngLE2LvjtRyayVAKMBchzmgnXXxC8c5/hvtRXgK4CJd4IAxxQE3a6zDsauUU0S46rXot47atWJV37MIsC1JSNJaJ8dlVkGeEMiCIKh5p0dmMQ8LQE+U1YmuCT6zOWYJvdT/0xgLeBbR40d2oKffRhTVigMViLc9hQb3FaSHDaJwMMNZzdsV4I6Pad3p+UPy+P1Cw8JU+hvIJ3dK1MEaxYg76cN+w6+1AtQ0dEmxO0Y6KUKMVmIgVhJUrIEq3wdIBrCOjiTaehFQ1NLZtYb1Ofj8koiYjBgkxE21a/aCbA3m/PrtXrYatetKbeUyT1Ec76OCDkBFlFiyx+rVrK9TbmootDaUuwizPMaCYBFDXmYTW6YH1GFcjHlfKHrFuNSk8bnS6rhW/MK25vWWNZtX95aCtFWV9WEOZwoTYfbWgON1mwMphz22dsoAx0DQbEP1D8VfJDGhISn0oHk7FGQdyQLfKgWjG6psTzPYs3KOHZlTDmDnmjBj2Ymiol2BGlqxBlZqB1nRCGdBUvYjBEfYBGnLnDCCal8aULCgmBfDLkDA2Ep51tRkvMkj4qaIj8RTEmazQNq8ADG2QcRqZNtfhmN8tFoK7j6YbPHNtmmBBeQ7yxaAE1XsnDDLlUtCpGpBpWQnCak5U8Nfg9iVAPsnrmMbUvZ0JNnpdTrJ+3uV9XuFqxEkHKJxLnGXXyb1+Q+TTtYb+UZ29io/gduLr3df+oOsz//d0fyfxKmHlWEBcT5sIwLstrhjHQhxEFCMcQmvrOtg6LrqhBr1ZPCpRqfFJgbq/qhLRpRUpjeFoLg+PzXs9Pzw+PBRXB8ckGI4VdnRgnM3ZLkT8+PDk+DX88vfhl+PDwamGMr+HyBvXPQlWT5N4fDgezmqVK/25Wx51TLAumkSRR/iFF0hdF1WGTlBOAA1MiQL48l146QPl0c/mqNkXowAUW6LyrB0PLF4O3J2eB4zUpKYdYqfjj/fDIY+uprTWPleXoXkThe1738vEa94i7IeWZXPb98X616W+0UzbdU8/jw8nC9aVaWww+Dy0Oq9tfh+ZlTT7bXc01rcsUGixs+5xnsTTKGs88fzMUSaNGwZBbOyDULItG8rEpzXbc2Lp5SJi8vDofvg+Hl4buTs3drNlBkYX5jqKSiUBNgPU1UkBXfWuFrNu6H8uXgw0fZtdUbrPfFyWcFG3ONpRMjQi4xLlE3fvfxEzsS1jX2HWh3ZBMbipmaa9YCMheRdFgdvf909suwDeTaJ6IENAoNWrKF9SCt1T46/TS89G8WHbq6wVZVwSYHF94+tVZ7uqFSq/3hkP55e3LqkDyrtmEMrSfcPHAXOfThm9VbJuH0Y0/FLNAyIW9L9rRstFMod0SevzIjn5NEleD408Xh5cn5WTAcHEFve692+juVEAAiKpuDXozWtwW0cMuXswxFmZxfoz+OdVKQHeZAdMZhHCagkaUzJuSDIo2BJuGr70BLudcdYCfb5wpTg9MBcqXLkw+D80+Xcgwvd+QYXrDXVfcxByWGTeXoWWeR8TsYQQ7dAfNk4RhkQuEkzLEU7qhXW3M0ZyOKgrwbJvkknZIYK5YqeD84vLh8Mzi8tPrffQ3dv2AfQKVkYpVgWrMYeBIJnlMec9BlSbRjNyC/F2MeFiQP776Gvfvh8N+Dw6NLJAtqof56/mYIDb+q4Srtm8rwkJYA4GjBsZN9BnPbGpezGdofywVaEl+x39IxGiN5ZZ9neZziXPkcB4XjBLLxcXD06fRQ9E3whd1+eirn9UrMSyiQOS7VzrdoVb4GeSZHU30M68hhHjBPCbYoZzCm62gMcgDZ8xd8UsYhKpEgAQLPiXkMkiS8FDiFqPeZZ+MUluQ0vSbC9h075uPymp2lRTQDuQpL5uwSvsYgMR4P3nx6F5yevxvukw8VxomCDhdOXzQOgvw4ODt8cwpUtiorBJddNAR+gxLVDiH+8D8A2h+wAG0ecs620aJ8CYCcg/R+Xe8bIPxoJSU3dL5vykDK31yZQCfpYqlYwL99GnyCIb5p67D2nVWbvqmaS3qMyjjWE2nh67Ewvg+XgCsw7pgGOBwes+G/nUaApygjIgrBOsJ2QCuMMSfakVhmC2rSpGZhFIP8l/+ZkVRzH0Gr2LI5TZ+sq83FnJwh4Wql7CaFt6BcCMeOKV0n/N70uJ+lCTc8B7rPYAEwhN2BKC3to8Baw2tOekuO8AgZtAglpxHQhTuOAq/Pp2D42a/jFAgds1mwlK3gD10qdEXX+p0QSevflZxp9lOtviHC9FgldPSYwc5d7g/vamZhNO7hW3VhjQVBF+a26lnL1mP6AlZab8HkivlDD+RHV+1TjThal60OqRZE8SerJFTpuWqJXvk5qole/xnqiV796SoKQffpagpW+wpVhXptEZbx+7PFXqz8NaIv1X+6AKtXe54Qq7fwfEGWYGfu0ifwvXplnszDtN3u8WC2UnosQHE4wi5WVXo2ca2bWEllexppfQoB7flWW/R71WpcBWjOw1sO5CrvTHvClRGktwfoadfMhmva5HJeWNwRIOzlmtJCkOTA0IEeT0DQQ4/utNY1xVhGib0UX7kM/9gl6HkNYD1mmD6qn5U5o8ccA4UqVA/O1oqrN/qYGlXBUaIQw0SKNRCiBRkUIrxAnW5CfvnJEgT7ebi4AfE9B5n7V6G4nILQV0ilRC2pGh+qWuLL6fmvAegOuKpumR3UV3YANaaRiMP4dHZ68uHkcnCM2stERgWAjrcF2oEmRcleJbjr9hr0I5L+ZFXgdXIqUL6KLu1XE+z4pgNgg0EENIjA2xJKjDgZkhxZmaBEjAryRAMiaTk4GdBnQW7OosltzPU5QWGQFnNYkLVGaU5fSu6X0ZynZUHq5JCA9mtYTG6m6TW7KCm4pJoc4OaFraD+DPqxVqRevcv3F4Ph+/PT4+DNRwT17s4O+x7+u/eqT6uIv395s537K5tK8B51YgBajOVicHkBmx2NBYzcu1ERgTz5A/wECU76kF6CiA0KJQsLtC4XudJYtuZ8jq7fyzC/ZZdZKFUOADqAL97CKGMWo7w+T6HdNCM3dgGFVZQb/R2grmIAHfSyWzTA4+dgQrEoSBR3RL/vhMgrlXnUubM0zjV8E1+CcFIIV7mIgBIS31ofGwckVJo8RCj9nZPFnoz3+J9al7GDoIayfM5UBYLRJEvzfGsBCjj8msNE5sCjo3EUR8WSkMkMCQlnvDAVGCWZY+cef8tdNOWpKuvxgo1G29//6/5o41/+8p9Xmz0aa4/Kdf1VRqP8ByrHqoLKaWa4XPDLl31AuSscIUyYwNNlf0E0ZKg7UhEFzfESCWAgLMXCHNMhVwhKLAFsTPLvtMRmS1cLNWMHll1iq+xNpOjthxQdsCA8C02aQtaNCqBopvQ9VuZ3Xlnk61CFGYY0SD5VoBlnwbM5kJgEY8Uy2BTYS1SIRuJln13eRDmrrFz62P5WwtZCLM7LOXGG5rAyFyq2Nib9xG5Bj0NOCwBs5mYeltok7pRJBNKkcrfOlO3/AQ8e9PE/rzrd/g3//cv+T1ePwYMaMtpnaA95Bv2oAIAPwRIQJC9sgdYzSG0w2hBf0Pqj/c9c9BQXXSw4hQOSRF1XEwck+riqnlH2tJEZfX0ClpTcChwTjU+j/NYIK7PWTWvIFTkFWnW8nTlB17a4wUKQDXWZA2hx0YFF+kJYKDbPFTCuDN09+yCJlLEIxxPbk9FiEtl6cEHwuM8e+KMRtu1Kze5029HUnbqnvFvcG3++lihutWDEvE5iHiblQvRK8WoB0KNwSSwAhL7q9X51QuYLFLiSckrXQ7CGkzDJ3ahS1vE7u7syBBs6ZaCw4c4nldEIYJ1MgOYnBRnNVAwrWgo4LGMdVFgHybIoz0sRN06mw+ubFLbXJF0A35fkjaJogfdohA1anLMyR17vId5++lVDCM3QJLup4zseXVMZj/DvOuRNa4MC3VcGwHlIoycIzrRUmYjbGg63UrI3W36BZmiQZJJcW8qpMlEjNJSdRV9TgWYLVfNA166BNaPUAIU72Wjj/whl/vvRCPCj//3/Qi4NJU7enZ1fDI4Oh4Nur7FqZeHB2ohWbfWv6hmhhR35Ws6zO+HLkT4HuWH2VcGTWW2lmqYAUVybmxDWVZkgcE0NQxP2nk9KTux4jHS/5tmIBL71QroYQFtPMXk0I4tszQnwxe3vdlIFpzX0Ifux+pCNdY0zEhgVCwJj/fEvbMdPGitOXU+xiUGrR7IzdHfMaP1FJz0FPE+Vml0ciXW9Yhe0NlN5qK1aRFodjZoBY32Q7T5qIenVWuGKP930uGLJsJJ30Xy9acvW3FnL2lFnzasnPv+Xrx9206tB+pw1NGbfvJBYrOb16wsaVa9SyBAnEWqFkbY6Eb5ampDxYFiTT4UGiC9WRCuK4DhPvHlj+KgTZE6ArdqxcckNMMcRQUkcj0mwXSEIvsCqF5ObjhiFBy+8S4UOBzwJQ2yDliTMimgGiizCC5t6dFAWH69+Vc3MLa+g/cMB2/W0BirxbcVgVVkTvT3DP5IFywV7kJUeFQccb9dyjEcjE0dtlJv0okzE5CWDwZMR83RaQnF55BiGlZUJYVgkfao+9+YKiQ66A/a3EafX1zxrPF8NmoL/TDVab40X7aeqrbPUvUqIlBYmqryFYhhoEteV+/M7Zf0YJQG8d+0WF8pwAQs/Lq8D0LTGMZ9WznmkMMidB+I9SokqJJT2WJjjC6iJvpkeBlMCIwZSgN7+KepR0HLGJ4B92D9sXJzIF5wHSr80l6sraesJhG3yXfD+ZHh5fvEfIjZEjrwS7f3yc2V5QVs5jqdjzMQUr8VkUJwAfYv+vJNBDFQVIUihDCEZYMirLDAo97mHTciJT3RSoYK4hvlmYRg8jk6NVdtrdUzEF288xBVFlu7iIcqZ3SgZUChIolXbEogl8gM41AtzBtT9QWe+WeKzpg8D1wZjt2lp/FaaC5KFc4FyMCv/mmCIihiDsRZSJbNHWSOFwqGOB4dM7FCfQJ/J8WRViLabHNAW7d4kYBLnQPRHlGYd3r/uI5UxTsF5McVA5FWYYhSGFVjITAmUrgANgmERIAXJi3C+sA9XSngoKtOv/kjS+w5Z5Wb4szPa+Pb9/rcf9r8dGskQgPRlAZqcUugfo5Q6Jox+jRZ0aHcJsEmQkLDRCE99TdDgKsgXiOTA3zAGJWYUTEX0FIAqd9X5kCnLZt6S3SBAmhagqoqDhV4A5X8QJkb2Pfv5Nf2g1+v7UWiOWrsF/70Q5kPLKktKLZWhGBJQ7tISCHBYFilAX8aeZBzmMUEYlAlPJumUKMqnJML4M5BhwoxO58X8OpwsmQRq25SBZ/RFV/16fF3v91lc5jcdc+Yd2fUgof9mWQp09nxIf6w6EzIBPAPhFESTrFN3g5IvzYz87z1hsBAadlnMtn7SCQ0NEEGLg0ZWA//0qTIHMjcByQrHkR9ggAXCDQ9f9qdcfbfEDQcUVdPNBQ2YPJFExfyOx0G+nI+BKtMPn9E+vkPWSJ/75WJRnwVBazZ+Q9p8cvb2fLThM7uPyr3dlz+PyhnfmSnQ8VivS5S3qfL0p5dT+O90Z+pUBnEGNsmvhxdnImIO/zo5e9dw3ALG8WO40zaOwcXF+UXjJP70auKvNvx0dDQYDpsr7ry26Da93tnbw9dyf/J5RFRbW4YeK8Jr+dccfdvXXP4CKjIBQoliZSW6kHnOZ1c7QvOD4ifYDQZXiE+HC6RnudQdYMtHlQtLRGj2NHKAghbxg4hYjDhPTmOgz3Kne6xdjTRfmfHr2VCbBAEDMyrTmCW2WRqtYV3Cp0D7kId5mPasJRR6MJsabVSlYVlZdcy+/k6DxG/0h/MZVo4qhtfOJ7mU+Fn+6dZGSRmrIwsjxqUN+VHjFpqs2Q9pMTseVQ5dTnpRVL9t6dOCpdHyIl10dgxTBcwrEOGnqNU8wM/HKyGcwZ9KItPppFiDgmRAUSV/vGIPLgHqPrIH2Tr8JQH0KDaKauwFqU8ZsJnrVDFj4rnS1kaohB8Sfk9oS24mDWw+hu/nwfW4BeNNLJsNiqOmvG7C0Z9bRbdzOOeejOa6q80ftL3SBa6xURUpYoiEUfGyg4p/ddEqMWs6adpvm7peco2ELjq3STBEetnxEzWTcr2Prm+2YBnTDH2/iRZbzeaojhZoL+/IwGBabo65Lq7DbBpjuHc6k3K0UM1M8bSmtYptEaWtxmPQVwV1KVqXE4wyWmsOQ1F2zdE3D7FiL2uPUqgd64zxGISfCJWoaRReJ2kOQh5tmI6CaYoibKSgWWsjzcOVJLttsIJVqdHeh1my1mCVnYx2swnWtYGphIU1IUnS21qDI3nzq4YmpY+1xwaCq06lOxTvs48OexEbQ1Kcz/7gVUNFS4JaqiRFeGyiOi85hUZMMb6Rr0spB91DBt/5siXGeNU0BxJ1/ROxY0jUuGsqNS/jIiI2gEEYgCfpzDOr9WZQu7m843dEGP8J8UraOxPSU2KcEYGG02wKA19ytOfVVXEOdtorO7eRzWZMfsy/bMKbzSvFlNVvH2euelQCRNWCkoCwHYtVw2f6vXll8mt4LxEXKhn8opZ8kZEI8zN129VdyplAZXNrEbPPNdkUYXkT5c4Bhmbjgi7NUD/KnnnDY1BoZJzT082a5BfxZpS0kkiWWRxH437G/1ZyDEg1XxOZsV8uYNJaUspyvMhSZChrmlA9xlORo3JUj1hZw+RPYdzV9+b1ogyiZJYKe4tJR4w1Oknw8BTI2yJWhN0AJ77H/FGIuGefT45PDumAKJYy9yA2bwrgow3slkKiYDlg/aGeCPgE6qgXu8vCeTAfY6kd4wPZsDNRW/Xz6LM9YJjigQbbvki7AK8XZdExN8mX0UZyFwGj3MrnkVA4t7ZgObPlFoz3AMfbEwjap+jAnhhFIOMpVQ1BsA4m+V0vSW84xjLDH2WCgawbV5YqgPTwwAzdsGLM8FlgMikkGAv1TfhfKCdRWfTzRRzhQbMe5nFyNAOqDSrBAXtp20xhab7oq3FFdjko/mXHTp4oilYrgiXR6SFK717ZrkNRXK1T3e7e1ZpmLY2mYGOa0TUBHjwJwjsQbFBUaTC/UvapHGFw9nlwdsRu+N1k+2bvx1dCXMbDiYi+MuUaSUKw4UAAkqSi2Z7VjlMw69lsvuDXEh9Ub7j29XL30PgGVOFAa+Z48Pns0+mpj6bi4MW8RZpBHALZq3BC1oeV8PWF+2DOpWBW5kKz6GjeicbzapdZiJivXFia5xTjUtFiUFt70bJIHLuKKcHvoHbiaUx/YqWWBai2g4ajk1I/K6l0rVrNqkuKlHhYnpKwwb/fuBXhte2npF4VD8XvlpPBOwCzXLVpMS4SQDeVe7M5cEycNntiVk8tcmfh0SfXTNNZb8M1TNCTGGPm3hOzFaTcY6g6LEExSud4WhhZxQWFIGcq1j9B83LtyMyjKR7kFamSBRdnKmWktEEdy7OPOfsQAlDvMPOllomoBxJHGM3VIeQ9Pd6fQoXFKWFl+hEbNadmKu5GtENYwjIpkGIUEfqUp+oYeW5kTdRH4LGVoVXm6Pzs6NPFxeDsUh7MwGB0ZXQgYyHwZFAbgk7O41lPUqQgmvp9hTDPFARNPB2bZgEIF03FhOdX5NYKRAR3W+wePjiAvhIShDFdkxm0chhqh1+QMup1TPbSl0Zy4FVCZhBEEm3m9ZdLnsdh/UFxPKJ3Z7BEunBLfVXwgc7rv6H8TJ1WCx6qAapsR48NgbpO64iXWpCvwHHWeTCnualmCXJx12lDnHjzr421eGq+/Uz8MdqghJEYe2AWFOTLX1noAcKRbAxE7i/DH24VEQcFAjqfvy9EQmD0tZBpa5s4h4edfYlju+KPx5ZGm08bGFVmZLgIxB4NYKIx8hU5IhJ7Z3EaFtT9o2k9vBAZjoBwgWA0iQqZMUlmlcOQlCxFCqD5fauITW17iHwJ9lsFYztNpMKRxgoWv1ApgK0QR7O9nvvBCPBpjgntyRa6T27CTM1WtWM2Y4mHdLyD0i9X0/KwFYexTZoyfzUzN4KdcOj7T9rbPjf90YJ57GdNroiPGT39gpExE7A5La9vQARkF4cftjDyXeWP3qcEBNtTfred38wpybESXgnFbsIcpfPXOzvswxuQpjiIpXjqQievc+U2h+WSDTlUhiKUA5TQqfcDO5m8L7+eaLfrSVhdfVvL4g1TJalYhOph/0GJZoK6GS86YLU+zVgCQJz3kv80rAFNVhwLwONZVpZhmaFvXtnINROAN16sGX5uTnd8tGQQzxiZckcLHMbx2UmZiPNZSda6qyayBvr+cwbngdY/ZyC6T8YzAtcPU8tjKslGUPNxaoIUI4OfazPVpT58Br/fhJjZ9o5TJA4mzpQ510l61Tj5Jqaar47IszEPJxhnSnEmWge7fUUIfUfwhXRAnxvzM9VtYbrm5gTZ/W0Mk92WuQO2tcEJwgFaDKVT6rcUrPvCxPUk1ZvH/bSc9Q3ZD79fp3XMSqpypzstNaSJXq/l132V6FSmNbVb1xnnek2+Rdt/rueycS4lMa9aQa1ZHKoTWvJ0ex7lOSZ8p2BSlTW734yG9dUDC5EjiDwIUhb9cmUKUnjlCOGQTMAt8s+YCi3uU5kc1IdiG/Xm9X62U4UutJAdtfvxhwzTcTTlhYfSWXNUSvuia84OkF4kRMp4lRTKmF5DO02ZLaU4JVKOGMk23KwXeq7753XT74t21+5OnzrswV8SNNOYKCwuKyDvRZ0cVUYUazB3qbOKNca+jUAV9dRRxw3ybosku0ZhS2Zdo8aKjPHq8ci5U3GoK9Hn5Bd2fcc1mhLGEyI05nFvloe9B7ZEL/K2o1xLru5vY81kyOoBUu251EbktG/LUy2hYomZ2FhbAmACTPsOESIfNrT+hvhKu5i+mY751rRETRNtmfJsGFk7M8zxq8noPCHzSKHrgvLgrqkOWo7BycJQrSTlNoeHiZo9aaYmtgVQ5nRWN0/goLyyIk/64XTawbIeWDmjVksiy5vjl8wDunPqWX2j7j4ps1xISWPPXV3ijoDqTFxlbNaX17XbMveSIb/BVgzVmvFzTcZV981m4+aRuGXXNB/LVV7DhIxPu6a9wpSMz5MV50Zjt3E1x9NM3nWMlKhNGc+eERuFDx1IPSDXbx/tCp1Zw8zRCHdAxWWix1o9aFQ0YbrKdgcyVwf+7pqXxdwUxcIMcLUfCjwPROdVG4a1sLnuC3ZepV2MrhMMptvd+1N/B/63y7bFqX06Eh3mdPoZCJrM94XkB2lGBtrIja6tqJwPrTR/HYmBgAIzqAakLvRS0yULcDVE6+tKFtLIW2vYyIguPCxmZFYU6yoVsW/YcJlMbrI0if7Oq7MlnnRwvkccPY18+33Vb7ktfJxLA3TVgW7QMU7eulcl2k87SdAA2myEq7pqb+cJtEM9pvHNflQSfIUVf4D3S2tWnczCV+SnERhRp8WpjAJ1NJNWfTbaeDAdFo/V1XYzy5VhEgX5GokXWfZ1x4a3QX2UuG/LZMKDIhWH/6RbCW1vZa7SzI42omnMTcojw/JEhPgkjYEVAMS2yF4nsxFJ4wTFjXvcYLqK37dVUUlondDsajYY4GHOz4mkrrwzVdnqjVNWjzppcVI59bQwFLtaHQ/h1BLgpUr0lxthDlQlqDIwW+HgTumMX+OZLQBq4BSuy2p+l3Ewkx4JGrSDq3ruEnnplqOYuWTblyRuo2eqAoom+tJOmhkk7Wk2mf2e0GtL0nOr77prS9ym+ATgLRIsT474LrzmRLvGPS2FrUaJunLtWoPBRX2a0zMEHZJupuV8QbeR9hjQgyjBVCoHew00e+aeRFoNFg08sxzYZgdaQQRMO0187Nl+mDUr6zIoRSVS8A5wyyDDwO2MB4peZu100oo6wscmcNUlysVNWOjRRZt5RZVNEio4NdrxvAS0avlkhtm27sNc5nOfymOb46UhmhEfFEZEmf8toSTyRl6UrQnIdIXI3xXN6QbRArUvEfSwpQiPvAoaD0JqXbRYGp9AgF6YtzU2zL/KHV9pgxovIVtMIUSr/3KypVnD8gAtsAhjx9+2IpFqNeJ2WtFIb2pjj2qoy8wL69rsys/cOPobPwXT5+WnY37LmGjNq8zqMH5GnivDdlk3ZpWVmQ/kIZbRxvvB6ccBYcnMOKb4ih3aWxf3oRoh/BSbFS0ZuLHw5eaDhMTjZr/fdzQ0ESLhimlEeA7EPy4M7cgSnxUHQz7k/g1w+0oPlk3EvTnE7DhBfOemBhMgEJKvQwovqgRaOunTCZ6b/oL2flmgQuFIjP9fpvlHyjTPk1Bcg07zzm7uWz0SAr6cLrLZrxYj6nx73hafIVMEaMQJ5FkAKUXwZEq3OsiDRih17TfGVKlIPMO/7D/lg488j0ZHfcOEvb+8/Mg+ng8v0WLybnDJ5EgEAx9n6X0OtEuEpovLJO7SCC/aScvpLMawxzHmQZQ3ajibUJLvdvJjaMzqpbBZVWpwXfnxQYHHiL9zUQNmEki10WwaHzUjR6UUMPoEs946vKaL4/DgwIf071Ech9t4r0zn1wjT6ebs7JLt7vR3/szgxY+v/sx+//FVF49Rx/xXPv4lKrZfv/xT/+WPrPPL+8sPpz3gprecvQOZJe2yoxsgZnybcg/j/9gwnIVZJKs0uKUOJ4hXYkThQngRYL23hR7vrfLvWyJkduvk2NWQzQqP5k+MckPoRSIdnRbT3QDkSk3ISU/oqtQHtZLhVpfLgP5g4XHbulwuuMrx4s7RWWFMB28cp+lfyK0Er8XWOVBj7KnuDuS/tukeUd5qDX6SBgW/e6RGp2Vx8HKnK4yeuc+jHNEt8Hj8Ankxnl3f2/ElOMNnnE6XlEYYymOEY6dKCNEGNTF32jSV5TnvYFtu6gf9VFEfd7s4nGhlIpPjRmYRRvMAc0uPNpTfpyJH7kikIETHNi0xiCjLA2ZSnPJH5NT1vsWcXDhdGLublstLDXyT+XRx+vy5kMYiDtdP6SREVQLN5Ly66PRpcwYkTkB0rpKH/kHTlknUabbr0M5VeecqCH3F9MW5Ymv+dY/OxNeccYP11CuYVhZUr2p9UeuiwvxYmRxpn1fHAAhPDz+eEEYIsfINiYoOM4NBBItwibvtf4wp0msOXE+9mfBdpoBMl5VXYNXNCh2Hf3dttSYjR7aQ0g1paLSxHS6iKi5KIYHIvVqthZnfIqPkTGiNyKWnTRlXuyKNSG0cNZK04AOiWXQtkinW1dVLCeeetVwWUkvQqdwAFuwMp1E9kAbQwa7dfFDdP25+Y6WbwOeF4WAyg+RkHrxamyf9yRNAiNwnTXJLQppDQSPCXQeK9t7Z11Rxqu4nrutY97U4ii1MBK+zoFquwPa8UCJZ055LAzeuQ45aQuVXxBNd+Zs2goGawoC0EZse+dZIIPWs54oz4necMKA1GpADNO4FV+wCd5gNa7vMml3gg6Efz4wd8ozYOk4AbT9lLPgYKIoNPK16y4kC9fzhTk60/VSjXtFaw/iUDldPntJxa7vcPRCgfW0CsgFMrbxru2obfzPSt/qc6zYbcGhNFjjZ1Qmwkh+0uzDEMaKKvoMkVHftT0n7LKezJ3K+SUwdbYj7l4XgI5iQvGpWJjT8swQ5iUcmTxGG9rGSjwx+1G6ZrHwhjqwnLIm1rJfDKGpfpzSJzPG+nUm+nhnEm5sAH934QXfvVHfC0tma6obbIqXVciTAWvqT40GrycOjU+CLIQ1eKenGo3ivK/tovt+e6qVr9063f8iPlvDTY54FoG5bnVrKqmzKWP4upHxVZ0v0I6RXcah31M8v2Xs1131xczh78aD3uEnH/ACKv292H0G+NL4tJsVmj+30d7r7/d3Z47dd9r+ZWWK2yLEEVIW/fPiYcdPpV2mRnnMcq+xsH9M4lkeOkfEmmBiyPtAlbh4WEh6eDnbwLUnv8fCjz3svT0mqs44e4zXazn573H+YAJ2h3ju/9diEkogtRDhn48nJflTwed7pCtp/z/4F63jtxapGgHckz4XF7jsgYXEJ8rFo9eBhs7cpmLc9aLzEhs642ZPx5KapNHfqw9gc9RL9a7XFDuxol+8qkf3gQc+q0sdLhkSciybVPz6YM3s0FcGGTavGuEoluQlzaZewueNv4mhyXfS3dExEBFHWLDrBLWCUrTYGFt/atcq/YJTnghLz1T4WmRYJ9BGBCuIb/x2WROCn1amCSpkUUayg0IBFOKoOzqgnBtsVO9Ph8BqCA6qZXTS7quiedLEx9T48MgI2n8ecLzp7TvdiPT2WCHzWVYRfsyNEQbyBXhIstSgascIVZD+w3cft+islh5FwQ2qWzthf07FeX6w/UbpNrV5UxBxa3ITS8G2z22Rdyfjq0DSVpaAmcj0muqXEWz1WzUH+TstCjLnO+9ETGTzwaodgPt4X563xUoL+TrOJZliO8c5GSQflMFAQQYBP+Bps+KuMMGpr7cvZOgW0/bRfA8EpZoIDy5pvnAo6qLC48dtnkFlXSjBWUpcUHNvOCi7MX8lL7AGlHbtEOpMr9uKhAsqjvnwdgcMCqo9d29pDSEjJ39uMNeKYWEguDz4VEkwYo5F8CdJ8giavrmki1AjCSiRuRko579yYMmCibJ+FQiLAyzQJCOMQaCqetZPZwKXZX9zbUVhZyf752PtV+KXD+DnotbcbyuTP7EI0VVPMaoREI2tKKLHIhvPzkEpaAEXXmv5CJ3fVFZliJJQx3+uPFX7xQJwJNjMfeTHqWF2pyrLwXmKNGQTrtczV1mlonYNwRVokWlMmhXJFiUMm2CLlPIU+vsoFawUlNfpg1RGCzW1gPn5/bKOD9b+fc9V0rH6//b2nxJp+VGNPrXJONnsjKaA7KJpk/3n4ezAthdIe5Bwz17963d+pC7hWkbV9m7uvn+Db/KbNt+nHKWNA2ikcbTtRnMpYBqUiK20KTBVnrdB+0G5M7egC5paCLma+tiG5wkK2yt0p9rYiJKxyr7FwhrGID3Z3jznor+QTaT2Jg88K4+G4nBl+5N29n2RijPYzIHSLZdl2xGmNzmmNZKZoaMwV8L2ntrQV918R5inmvyrMG5K2btdPTZcm43/chv4gU542IWvrrOPWbeO/GopKjlJhKnnUFS763bX/M2BZs3sRnSKYveTzKNbsW9bDHt6RXigxEjWadaK5B7/zSYlx0io73BYKsjGHrQ8UQWSZLLIwyemiFLwihZILDId40/xiWYnYGh/KpXy54FmUTqNJba7Me+KeZXE1KYgWSZTfQDf9+e2dcdDGjgx3HNzhvYzUFFhyQBABKRw/6BqOZgDCve+tY2tFWp2Mz4LJ7FqWlUIa3TKOSXVkIlrA2YdHDRGEHFi3r+Rf3RhViZBVKU0INvJViPfSr+lct2wBwpBvEUdRjrOzCPnSxMCmixLKZPYgsegxeKi7NjeZqCwyhq7VNEJ4Rdt166iPzOiWlGoKvbrLlrSS6j7i525u34nYZx+f3+2zt7yY3JAkLXMq7ku7luGpIPFiyvlCHgI3MqN8h8k7IiAAnHVEwseXO7r4NUvLZFrfI+4E61un69H0amHMlT7owzhP5WEFGpbY697ToBR1F+FtK3TjFqoFpq+ersOtr9CqUXn2XFz2OVk9cc+UfFK/R7wx7NmYUE+cbruW1yfJDSlinxGHcRcrFQ8LaJOx7i2pBysjldpzHDSkgaibMXYHCWYEJMwfP19QGMDejrl95Cfy4IfJNe+4tZxMf3iwBse8crz4WJl15SjtUu7epIJeEUp88UtN+LgUv9mt7ewKf8ozmogrL8KozQZ88/dW6yi4/8B2u+xb9gq1eN+EVlhr917OXrIvQ6CVrKKYeOFqVKh7VrUzMBMMcCY/KH1pMFJ0HjCvtTlAkLr7r7uP6MMIFzle8uA5xaGZpLG0de6kFVANOfXse2/Nhe3VjGtFyLpPqNSA61N9HJjKE1goyEinQgckyu6f4ZfI7axsOZp1wwclfJposWPgsAqmme/gUA0FLGDvFu2rB8XWQa8GPFG2QelFB6zqsyFNu86ettIM5IOQ6IcvfE4x/LStaOu2Zqq1w6inIALcKvufzwqmemnGIrkG1NJXIo9+NRsbpmU24a65WYAVu1xy0K4pqi6ii00TdTQoE0CThkYd68R93A341uLP+tKRltZa0OxemTYaGNeeYYkxmjV9Vk5bjYqtx4QyC+b6HeseDN5mHS09pj9MsRGX/7TzupnmYd/kXWcf3nTJfwgAF9EiUy20ROqYFVLb8YuUVO1NGcVT9vYtZsHXNKJJOp8bjCiHeXHERrSjHCgNQtl3tY9CykjK2MyrMslmTjV4F4BUiDVe6ieZ7wIcw8QpT4nAxTfRy/v+3o+vAdDvB5+PzOvX84CuD0BrM/wr0o+oZik1ySHW0N5qhtNQXluAtyZ4bjBwzyhiV678XrXj2Y93AWjcMtJxa7J/J6ajXxBA9xFQfqpC/Fi8ki8nfxPlMVAcINi1Ah4bMp76e4yj8e8EH9FyNtMbdoZwx7PlLMRsLlqfnv6eN3nt2oR/5ORfP2fyGl7Pp6Jd4yIJvL0L2If8tTQA9oLJ+K/PF4cf6uz1dOwEqSRu47PPx4OjfbzaEi+xRTsG3rtSFRmnRRHzBBQaAxWbgA5jBP5boPiNALi5J1uIVAfKaSg8eQ2l6XaViqxdmUqBQRW+Oag2fvsA7gSktbqN/RvNfC/W0fENbIXu6St8iyflgoRfU77IoMhVbrlbHvydZ6mvjopHk2gXLfj+rlOu0trr11cGWZiWmXQLkBu+o1lZ5Cdsf+8VOufdZI+nYZmAhi1psrxxRGquyFnrMLl5mkRFqkVsrmIrP4a7IPdAdcQizf41a2I2KHubbsDO5kM1HRkGgfEPdOCkOc0+65hihiOU4zSZcVPLR3JOADb05OXH+v0rH08+DlruZTFubxmXM+TNB7sGqEVGlzHdRNrk58EbKILFRMZRWO9ni9z7Pl9wOsGA9wvVyX3wcTUHusgbb0kDRoRTkPcI+yQ4LHVA//hz1OODaroooOUjgnEeNCcpW3Git56owGTRvLzCCJv9snulxtPi6JDGQAr4o3sUZLv+Gl+VVoQaiL2QoJVpgYWxfG0zbQClmCQ1oE1T/F53jGhTRDQMyuevGh56pAuffHNoWSUxXKJb0MQ22w12dnbw/z6B2qgk9gjwqs7PP/d3emjO6ezgH52qzW16uYsvJQ3skq6+I4lgU/sSqtCDBlP89Qchm1ZjzZXD56tRFJ+W6FL9QRMMFt0yaBZdEoYe+pauoagWWWLFdXvOXevPygAXt8Ja4VpuNVra/WphV5anhd2vlnVlebms+9qirq5TJQTTA6qbaz22YDApCVECTSYT3hFr0qMrPIXNULxRp93GePeNCPxszSyJz3rhkXy665rdGvg9dY8nDMzcRJ37KMHrn6r7lLp99jbjnOKtoM0mZV5/iL+BrCck2iZkV896h6mozfsQ6JsKn/AHmOrPc474YDe3URyvGvXTnDy+Z71547Omi6jpaXcQ2c8zgIZP+9kofBpDB/RnRVAbSrXrIrgS8PfZgyI44qACnlNQJGW/vzOj4wnsP9kQ6YUqTcTDf2yoGqspVwLhbuH9Ih5OjekACLSjO6lH47O7zRzZ5paeovWecZMVtFqhG3eBXB5xP/Mq82LdBYgBSD8aFkpRXX8wxNdbCR24PN1C6ItswEdcWjXpi++kYYFuvNNqEkfy0GwSp6/NJnEv5NGK+GrSBN5a+ZMegifqb+wLDB8vrZ1yWCFrunjV8f9LC/aC0bXG1ckHymsljoprFtCWgxaLdNHx9F7dg6F39Qrv6VgstbsJZdSaFiiiu9LrunpGOicDsRkG0m3OU2d7qioM69mxJGb8goyP9xi97WrNpu/20/ntdm/DOE2QQ1O4OvBQmcOfaodgwxCN53V0jm0kf90XofhLxzNk4YZ5jMPBBge8vQqmVo8/9gkf9aAk9G+mGSZdpMtjtGV5ksThly6eIEk0Sw1PTL2FT8P5U3kBOIXWBHEKW0vm9tTiwDyZj8VpKnFRoQgXYOh1gm1Ed13jsmPAANujv3OM48IvXTdgexrDGkZ4IzFQ3Pq9iHY1rmZ07N6rI71pGWrv80uPKFGnP7dWL8wrN6F2KtJzngvj6Dzr3DCxaoIU3+69ArLpDLld+ItcpCsZAeZWW3VKuda2fM5BQTgUEIwQQuyuQpID+W9TVt4/bpb6Ya6v6wbN92GyDGT21wNQDpedXM//RWidV2dH9Yb7d2Fc8txrhZN4abbfOtc1DpE3diPXqMlypaG+V1VrubFPq7tDQRtr1NNw/gcQh72j1vcFlEExTHv1LYgeTREv+LQpJ470ewLt7hsxL/VpZP2e4yaF2qIcNdEEitSQ15O4ljxryiksNqJ413QBugp6GnAkxoXLSC9lBYFbNoVstjuQmRH4/o8/uVx/lZnipx111xz0L6/uxTuEhceClDDzkLAdvtnSC3yXDOLkWLVTH01+Ujswuv12EcP+KLM94bcPb8jt131SjyJc+b3yFB6qSGbksuxh8z94vik90Y6zWhzk3jxLWQc9iCD5xhiF0d182pyPFHJMlmru3suxHyX6Auedl3ERJjwtcxBpJa99Up/HuKHYaXotT0LKy85wxoOzwzeng2PW+cyzcZrz7qaIUKQGAQxiL6rJH58MZfEPKDsVQEgxa3e8XB8KBlrrohqhprl79J1TTbz2AZF84rs4iiSUKrSxGcaOTMf065kv6S8Z0X8gThTa8lSPoRv1oIMvel1M0ggiXeLNZy2cE7aIISehYjoLOyaiXQJaTQ4m8N/ZLpKD2gQ5UOkHxCWHZVYTBC23mV/vbEqrrMl2QJGfJfNRSLS63J7RmdNZxnNPNrFPF6fPFBetElWkmOemUz/3X93F09bluL5uh6aprdM6a7LW2tiFW+QCbfH8PF6CQV9kyee1V9+yH9vYvAYhT0ofGX6oM3YHAWQq7YZ8+R391tFV9vQ2SlUfIn/1M7uMFvt4AUFYFumWQpgliOa3mOkbhAk8Qwz90lUBPpRFZSulr5vW/fWbjBJzcAz9gHrDgi/YXvOwNcnFFt3U06T3vGAfnTM3dG9NES5l6hJypIioEUYGIXEAHNgdk3QZuJBSLQRtNvt4mrguy6DKQaS8EHm72yVz0sx0cf7KoQd6u55uRRA89rKiID5P9Pnp/r6wsJKf+C6mrysafr+qrva2vbr0/1UV6bfIVNJaT/oBq3r0e416lT+wqinf9GTMxorqT3AN2m7BBiLmXyqjJ6F32e3pYew7xNCkLhDIm7Uxr1vH2rj17Sg4ZWGQ0nMzGp95goKkkKhgN1yD/IVH7+TpAyWLCAVhQGkNBZdGVJUyvBDqAek51Lw1lAllD+SFlNmc7uSEpVB1INsUsrw9sQPrt2dyB84b2YFpGRPdIctXrEiat37hy3EKgvhJAmQvKxd6qM5aRBmdsRIuUiGJVFPC7Vrm0AYb3pQFcRIyL1vcwHO6XIy3RW9PZzN0ODnkWdb03hPR4h+yXUPSsgckLCCoBoFwbQWEgUFQubVs3IQ+Nh7/L6q19YE="
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
coordinator_url = "" #@param {type:"string"}
drive_folder_name = "" #@param {type:"string"}
#@markdown *Optional*: If Google Drive auto-discovery takes time, paste the Cloudflare or Gradio link from the Main coordinator here (e.g. `https://xyz.trycloudflare.com`), or specify the Google Drive folder name (e.g. `colab drive download`). Leave blank for auto-discovery.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Coordinator URL: {coordinator_url if coordinator_url else 'Auto-Discover (via Google Drive master.json)'}")
if drive_folder_name:
    print(f"   Drive Folder:    {drive_folder_name}")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(coordinator_url=coordinator_url, drive_folder_name=drive_folder_name, enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
